##1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##2. Install Packages

In [ ]:
!pip -q install lxml tqdm pandas numpy

##3. Imports

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import xml.etree.ElementTree as ET

from tqdm import tqdm

##4. Dataset Paths

In [ ]:
DATASET_PATH = Path("/content/drive/MyDrive/Dissertation/Dataset")

XML_PATH = DATASET_PATH / "ANNOTATION_STAGING"

PROCESSED_PATH = DATASET_PATH / "Processed"

PROCESSED_PATH.mkdir(exist_ok=True)

print(XML_PATH.exists())

True


##5. Locate XML Files

In [ ]:
xml_files = sorted(XML_PATH.rglob("*.xml"))

print("XML files:", len(xml_files))

print()

print(xml_files[:5])

XML files: 1319

[PosixPath('/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/161-resubmitted-correction-3-9-12.xml'), PosixPath('/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/157/158.xml'), PosixPath('/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/157/159.xml'), PosixPath('/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/157/160.xml'), PosixPath('/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/157/161.xml')]


##6. XML Namespace

In [ ]:
NS = {
    "nih": "http://www.nih.gov"
}

##7. Reusable Helper Functions

In [ ]:
def read_xml(xml_path):

    tree = ET.parse(xml_path)

    return tree.getroot()

In [ ]:
def get_study_uid(root):

    tag = root.find(".//nih:StudyInstanceUID", NS)

    return tag.text if tag is not None else None

In [ ]:
def get_series_uid(root):

    tag = root.find(".//nih:SeriesInstanceUid", NS)

    return tag.text if tag is not None else None

In [ ]:
def get_reading_sessions(root):

    return root.findall(".//nih:readingSession", NS)

In [ ]:
def get_characteristics(nodule):

    chars = nodule.find("nih:characteristics", NS)

    if chars is None:
        return None

    data = {}

    fields = [
        "subtlety",
        "margin",
        "lobulation",
        "spiculation",
        "texture",
        "malignancy"
    ]

    for field in fields:

        value = chars.find(f"nih:{field}", NS)

        data[field] = int(value.text) if value is not None else None

    return data

In [ ]:
def get_rois(nodule):

    return nodule.findall("nih:roi", NS)

In [ ]:
def get_contour_points(roi):

    points = []

    edge_maps = roi.findall("nih:edgeMap", NS)

    for edge in edge_maps:

        x = edge.find("nih:xCoord", NS)

        y = edge.find("nih:yCoord", NS)

        if x is not None and y is not None:

            points.append((

                int(x.text),

                int(y.text)

            ))

    return points

##8. Process One XML
Now we combine everything.

For one XML:
XML

↓

metadata

↓

Python dictionary

##9. Process One XML File

###9.1. Create a Parser Function

In [ ]:
def parse_xml_file(xml_path):
    """
    Parse one LIDC-IDRI XML file and extract all annotated nodules.
    Returns a list of dictionaries (one row per annotation).
    """

    root = read_xml(xml_path)

    study_uid = get_study_uid(root)
    series_uid = get_series_uid(root)

    reading_sessions = get_reading_sessions(root)

    annotations = []

    for reader_id, session in enumerate(reading_sessions, start=1):

        nodules = session.findall("nih:unblindedReadNodule", NS)

        for nodule in nodules:

            nodule_id = nodule.find("nih:noduleID", NS)
            nodule_id = nodule_id.text if nodule_id is not None else None

            chars = get_characteristics(nodule)

            rois = get_rois(nodule)

            annotations.append({

                "StudyUID": study_uid,

                "SeriesUID": series_uid,

                "ReaderID": reader_id,

                "NoduleID": nodule_id,

                "IsLargeNodule": chars is not None,

                "ROICount": len(rois),

                "Subtlety": chars["subtlety"] if chars else None,

                "Margin": chars["margin"] if chars else None,

                "Lobulation": chars["lobulation"] if chars else None,

                "Spiculation": chars["spiculation"] if chars else None,

                "Texture": chars["texture"] if chars else None,

                "Malignancy": chars["malignancy"] if chars else None,

                "XMLFile": xml_path.name

            })

    return annotations

###9.2. Test the Parser

In [ ]:
sample_annotations = parse_xml_file(xml_files[0])

print("Annotations extracted:", len(sample_annotations))

sample_annotations[:5]

Annotations extracted: 21


[{'StudyUID': '1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051825667176600857752',
  'SeriesUID': '1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094259402036602717327',
  'ReaderID': 1,
  'NoduleID': '0',
  'IsLargeNodule': False,
  'ROICount': 1,
  'Subtlety': None,
  'Margin': None,
  'Lobulation': None,
  'Spiculation': None,
  'Texture': None,
  'Malignancy': None,
  'XMLFile': '161-resubmitted-correction-3-9-12.xml'},
 {'StudyUID': '1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051825667176600857752',
  'SeriesUID': '1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094259402036602717327',
  'ReaderID': 1,
  'NoduleID': '3',
  'IsLargeNodule': False,
  'ROICount': 1,
  'Subtlety': None,
  'Margin': None,
  'Lobulation': None,
  'Spiculation': None,
  'Texture': None,
  'Malignancy': None,
  'XMLFile': '161-resubmitted-correction-3-9-12.xml'},
 {'StudyUID': '1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051825667176600857752',
  'SeriesUID': '1.3.6.1.4.1.14519.5.2.1.6279.6001.3402021880942594020366

##10. Convert to a DataFrame

In [ ]:
sample_df = pd.DataFrame(sample_annotations)

sample_df.head()

,StudyUID,SeriesUID,ReaderID,NoduleID,IsLargeNodule,ROICount,Subtlety,Margin,Lobulation,Spiculation,Texture,Malignancy,XMLFile
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,0,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,3,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,4,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,6,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,2,12468,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml


##11. Dataset Statistics

In [ ]:
print("=" * 50)
print("Sample XML Statistics")
print("=" * 50)

print("Total annotations :", len(sample_df))

print("Large nodules     :", sample_df["IsLargeNodule"].sum())

print("Small nodules     :", (~sample_df["IsLargeNodule"]).sum())

print()

print("Malignancy distribution")

print(sample_df["Malignancy"].value_counts(dropna=False).sort_index())

Sample XML Statistics
Total annotations : 21
Large nodules     : 2
Small nodules     : 19

Malignancy distribution
Malignancy
1.0     1
3.0     1
NaN    19
Name: count, dtype: int64


##12. Process All XML Files

In [ ]:
all_annotations = []

for xml_file in tqdm(xml_files, desc="Processing XML files"):

    try:
        annotations = parse_xml_file(xml_file)
        all_annotations.extend(annotations)

    except Exception as e:
        print(f"Skipped {xml_file.name}: {e}")

Processing XML files: 100%|██████████| 1319/1319 [06:55<00:00,  3.17it/s]


##13. Build the Complete Metadata Table

In [ ]:
metadata_df = pd.DataFrame(all_annotations)

print("Total annotation records:", len(metadata_df))

metadata_df.head()

Total annotation records: 20383


,StudyUID,SeriesUID,ReaderID,NoduleID,IsLargeNodule,ROICount,Subtlety,Margin,Lobulation,Spiculation,Texture,Malignancy,XMLFile
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,0,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,3,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,4,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,6,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,2,12468,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml


##14. Dataset Summary

In [ ]:
print("=" * 60)
print("LIDC-IDRI Annotation Summary")
print("=" * 60)

print(f"XML files processed      : {len(xml_files)}")
print(f"Annotation records       : {len(metadata_df)}")
print(f"Large nodule annotations : {metadata_df['IsLargeNodule'].sum()}")
print(f"Small nodule annotations : {(~metadata_df['IsLargeNodule']).sum()}")

print("\nMalignancy ratings")

print(metadata_df["Malignancy"].value_counts(dropna=False).sort_index())

LIDC-IDRI Annotation Summary
XML files processed      : 1319
Annotation records       : 20383
Large nodule annotations : 7050
Small nodule annotations : 13333

Malignancy ratings
Malignancy
0.0        2
1.0     1026
2.0     1654
3.0     2674
4.0      969
5.0      709
NaN    13349
Name: count, dtype: int64


##15. Save the Metadata

In [ ]:
metadata_file = PROCESSED_PATH / "annotation_metadata.csv"

metadata_df.to_csv(metadata_file, index=False)

print(f"Metadata saved to:\n{metadata_file}")

Metadata saved to:
/content/drive/MyDrive/Dissertation/Dataset/Processed/annotation_metadata.csv


##16. Load Saved Metadata

In [ ]:
# Load generated annotation metadata

metadata_file = PROCESSED_PATH / "annotation_metadata.csv"

metadata_df = pd.read_csv(metadata_file)

print("Shape:", metadata_df.shape)

metadata_df.head()

Shape: (20383, 13)


,StudyUID,SeriesUID,ReaderID,NoduleID,IsLargeNodule,ROICount,Subtlety,Margin,Lobulation,Spiculation,Texture,Malignancy,XMLFile
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,0,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,3,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,4,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,1,6,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,2,12468,False,1,NaN,NaN,NaN,NaN,NaN,NaN,161-resubmitted-correction-3-9-12.xml


##17. Check Columns

In [ ]:
print(metadata_df.columns.tolist())

['StudyUID', 'SeriesUID', 'ReaderID', 'NoduleID', 'IsLargeNodule', 'ROICount', 'Subtlety', 'Margin', 'Lobulation', 'Spiculation', 'Texture', 'Malignancy', 'XMLFile']


##18. Check Missing Values

In [ ]:
metadata_df.isnull().sum()

,0
StudyUID,0
SeriesUID,0
ReaderID,0
NoduleID,0
IsLargeNodule,0
ROICount,0
Subtlety,13349
Margin,13349
Lobulation,13349
Spiculation,13349


##19. Filter Clinically Relevant Nodules
####For cancer classification, we do not want tiny nodules.

The standard approach is:
Use nodules ≥ 3mm because these have:

1. malignancy rating
2. radiologist characteristics
3. complete annotations

In [ ]:
large_nodules_df = metadata_df[
    metadata_df["IsLargeNodule"] == True
].copy()


print(
    "Large nodule annotations:",
    len(large_nodules_df)
)

large_nodules_df.head()

Large nodule annotations: 7050


,StudyUID,SeriesUID,ReaderID,NoduleID,IsLargeNodule,ROICount,Subtlety,Margin,Lobulation,Spiculation,Texture,Malignancy,XMLFile
9,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,2,12453,True,2,3.0,4.0,1.0,1.0,5.0,1.0,161-resubmitted-correction-3-9-12.xml
20,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,4,2881,True,2,4.0,2.0,1.0,1.0,4.0,3.0,161-resubmitted-correction-3-9-12.xml
22,1.3.6.1.4.1.14519.5.2.1.6279.6001.339170810277...,1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102...,1,3,True,6,5.0,4.0,1.0,1.0,5.0,3.0,158.xml
23,1.3.6.1.4.1.14519.5.2.1.6279.6001.339170810277...,1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102...,1,4,True,6,4.0,4.0,1.0,2.0,5.0,3.0,158.xml
25,1.3.6.1.4.1.14519.5.2.1.6279.6001.339170810277...,1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102...,1,6,True,8,5.0,5.0,2.0,3.0,5.0,4.0,158.xml


##20. Malignancy Distribution

In [ ]:
large_nodules_df["Malignancy"].value_counts().sort_index()

,count
Malignancy,
0.0,2
1.0,1026
2.0,1654
3.0,2674
4.0,969
5.0,709


##21. Create Binary Cancer Label
####For deep learning classification, we need a target class.

A common approach:


1,2,3 → Benign

4,5   → Malignant

In [ ]:
def create_label(score):

    if score >= 4:
        return 1

    elif score <= 3:
        return 0

    else:
        return None

##22. Check Cancer Labels

In [ ]:
large_nodules_df["CancerLabel"] = (
    large_nodules_df["Malignancy"]
    .apply(create_label)
)
large_nodules_df["CancerLabel"].value_counts()

,count
CancerLabel,
0.0,5356
1.0,1678


##23. Save Clean Metadata

In [ ]:
clean_metadata_path = (
    PROCESSED_PATH /
    "large_nodule_metadata.csv"
)


large_nodules_df.to_csv(
    clean_metadata_path,
    index=False
)


print(
    "Saved:",
    clean_metadata_path
)

Saved: /content/drive/MyDrive/Dissertation/Dataset/Processed/large_nodule_metadata.csv


##24. Load Clean Metadata

In [ ]:
import pandas as pd
import numpy as np

large_metadata_path = (
    PROCESSED_PATH /
    "large_nodule_metadata.csv"
)

large_df = pd.read_csv(
    large_metadata_path
)

print(large_df.shape)

large_df.head()

(7050, 14)


,StudyUID,SeriesUID,ReaderID,NoduleID,IsLargeNodule,ROICount,Subtlety,Margin,Lobulation,Spiculation,Texture,Malignancy,XMLFile,CancerLabel
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,2,12453,True,2,3.0,4.0,1.0,1.0,5.0,1.0,161-resubmitted-correction-3-9-12.xml,0.0
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051...,1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...,4,2881,True,2,4.0,2.0,1.0,1.0,4.0,3.0,161-resubmitted-correction-3-9-12.xml,0.0
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.339170810277...,1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102...,1,3,True,6,5.0,4.0,1.0,1.0,5.0,3.0,158.xml,0.0
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.339170810277...,1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102...,1,4,True,6,4.0,4.0,1.0,2.0,5.0,3.0,158.xml,0.0
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.339170810277...,1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102...,1,6,True,8,5.0,5.0,2.0,3.0,5.0,4.0,158.xml,1.0


##25. Check Duplicate Nodule IDs

In [ ]:
duplicate_check = (
    large_df
    .groupby(
        [
            "XMLFile",
            "NoduleID"
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
)


duplicate_check.head(10)

XMLFile  NoduleID  
102.xml  Nodule 001    4
113.xml  Nodule 001    4
089.xml  Nodule 001    4
039.xml  0             4
081.xml  Nodule 001    4
154.xml  Nodule 002    4
159.xml  0             4
076.xml  0             4
         Nodule 001    4
191.xml  Nodule 001    4
dtype: int64

##26. Create Consensus Groups

In [ ]:
large_df["ConsensusID"] = (
    large_df["XMLFile"].astype(str)
    +
    "_"
    +
    large_df["NoduleID"].astype(str)
)
large_df["ConsensusID"].head()

,ConsensusID
0,161-resubmitted-correction-3-9-12.xml_12453
1,161-resubmitted-correction-3-9-12.xml_2881
2,158.xml_3
3,158.xml_4
4,158.xml_6


##27. Aggregate Radiologist Opinions

In [ ]:
consensus_df = (
    large_df
    .groupby("ConsensusID")
    .agg(

        XMLFile=("XMLFile","first"),

        StudyUID=("StudyUID","first"),

        NoduleID=("NoduleID","first"),

        Readers=("ReaderID","count"),

        MeanMalignancy=("Malignancy","mean"),

        MedianMalignancy=("Malignancy","median"),

        ROICount=("ROICount","max")

    )
    .reset_index()
)

##28. Inspect Result

In [ ]:
print(
    "Consensus nodules:",
    len(consensus_df)
)

consensus_df.head()

Consensus nodules: 6407


,ConsensusID,XMLFile,StudyUID,NoduleID,Readers,MeanMalignancy,MedianMalignancy,ROICount
0,000.xml_0,000.xml,1.3.6.1.4.1.14519.5.2.1.6279.6001.213637791937...,0,1,1.0,1.0,13
1,000.xml_0102,000.xml,1.3.6.1.4.1.14519.5.2.1.6279.6001.261049959740...,0102,1,3.0,3.0,4
2,000.xml_1,000.xml,1.3.6.1.4.1.14519.5.2.1.6279.6001.213637791937...,1,2,2.5,2.5,11
3,000.xml_124130,000.xml,1.3.6.1.4.1.14519.5.2.1.6279.6001.213637791937...,124130,1,1.0,1.0,12
4,000.xml_124241,000.xml,1.3.6.1.4.1.14519.5.2.1.6279.6001.213637791937...,124241,1,1.0,1.0,7


##29. Create Final Cancer Label
####Now use the consensus malignancy score.

We use:

Median score >=4 → malignant

Median score <=3 → benign

In [ ]:
def consensus_label(score):

    if score >= 4:
        return 1

    else:
        return 0

In [ ]:
consensus_df["CancerLabel"] = (
    consensus_df["MedianMalignancy"]
    .apply(consensus_label)
)

##30. Check Final Distribution

In [ ]:
print(
    consensus_df["CancerLabel"]
    .value_counts()
)

CancerLabel
0    4989
1    1418
Name: count, dtype: int64


##31. Save Consensus Metadata

In [ ]:
consensus_path = (
    PROCESSED_PATH /
    "consensus_nodule_metadata_v2.csv"
)


consensus_df.to_csv(
    consensus_path,
    index=False
)


print(
    "Saved:",
    consensus_path
)

Saved: /content/drive/MyDrive/Dissertation/Dataset/Processed/consensus_nodule_metadata_v2.csv


In [ ]:
print(consensus_df.shape)
consensus_df.head()
consensus_df["Readers"].value_counts()
consensus_df["CancerLabel"].value_counts()

(6878, 9)


,count
CancerLabel,
0,5224
1,1654
